# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook demonstrates how to load, explore, and process the FAIR² dataset of extracellular vesicle protein mass spectrometry using the `mlcroissant` library.
The workflow includes data loading, metadata review, schema exploration, loading records, basic EDA, and visualization.
### Dataset Source
The dataset source is defined using a Croissant schema at the following URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset object (metadata & record access interface)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)} | Version: {getattr(metadata, 'version', None)}")


## 2. Data Overview
Let's review the available record sets, fields, and their `@id` entries as defined by the Croissant schema. These identifiers will be used in data access and referencing.

In [ ]:
# List all record sets by @id and name
print('Record Sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    info = f"  RecordSet @id: {rs.id}"
    if hasattr(rs, 'name'):
        info += f", name: {rs.name}"
    print(info)

# For each record set, list all fields (@id and name/description)
for rs in record_sets:
    print(f"\nFields in RecordSet '@id': {rs.id}")
    for field in rs.fields:
        desc = getattr(field, 'description', None) or ''
        print(f"  Field @id: {field.id} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)} | {desc}")

## 3. Data Extraction
We will now load records from each record set, referencing each entity by its Croissant `@id`. Data will be loaded as pandas DataFrames for analysis.

In [ ]:
# Collect all record set @id strings
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id={record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2).to_string(), "\n")

# To proceed, select a main record set that contains quantitative protein-level data.
# For the purposes of this notebook, we'll use the first available record set (if present):
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main analysis RecordSet will be: {main_record_set_id}")
else:
    print("No RecordSets available in the schema.")

## 4. Exploratory Data Analysis (EDA)
Apply EDA to the main quantitative table: filter by a numeric field, normalize data, and try grouping by a categorical field.


In [ ]:
# For demonstration, infer a numeric field (e.g., one containing 'abundance', 'count', or 'weight') 
import numpy as np

df = dataframes[main_record_set_id]

# Guess a numeric field:
numeric_candidates = [col for col in df.columns if df[col].dtype != object]
if not numeric_candidates:
    try:
        numeric_candidates = [col for col in df.columns if np.issubdtype(pd.to_numeric(df[col], errors='coerce').dtype, np.number)]
    except Exception:
        numeric_candidates = []
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Numeric field selected: {numeric_field}")
else:
    print('No numeric columns found, EDA will be skipped.')

if numeric_candidates:
    # Convert column to numeric just in case
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    # Filter rows
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a field containing 'description' or pick a likely categorical
    group_field_candidates = [c for c in df.columns if c.lower().startswith('desc') or c.lower() == 'protein']
    group_field = group_field_candidates[0] if group_field_candidates else df.columns[0]
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
We can now plot the distribution of the principal numeric field and show group statistics if available.

In [ ]:
# Simple visualization for the selected numeric field
import matplotlib.pyplot as plt

if numeric_candidates:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped_df is available, show top 10 means
    if 'grouped_df' in locals():
        grouped_df.sort_values(ascending=False).head(10).plot(kind='bar', figsize=(10, 4), title=f'Mean {numeric_field} by {group_field} (top 10)')
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a complex, multi-table dataset described by a Croissant schema using `mlcroissant`. By referencing all entities by their Croissant `@id`, we ensured robust and reproducible data access regardless of the dataset's complexity.

- We loaded metadata and explored available record sets and fields.
- We extracted data from a main record set by `@id` and performed EDA on numeric and categorical fields.
- We visualized distributions and grouped statistics.

**Next steps**: Use domain knowledge to select meaningful fields, check the Croissant schema for detailed entity documentation, and apply ML workflows as needed.